# HUDM 5001 — Programming for Data Science
## Session 11 · Classes and Object-Oriented Programming
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cgpan/hudm5001/blob/main/Week_11/11_Classes_OOP.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)
**Teachers College, Columbia University**


> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.  

About this notebook:  
This notebook is adapted from Prof. Youmi Suk's materials for HUDM 5001, Programming for Data Science ([original repository](https://github.com/youmilab/hudm5001)).  
Wording in this notebook was refined with Claude. I am responsible for its accuracy.  

Version: Nov.15.2026 6:00 pm 

Last week's notebook closed with a short preview of classes. A function packages behavior. A class packages behavior together with the data it works on, and gives the bundle a type of its own. You have been using classes since the pandas sessions without writing one. Every DataFrame you loaded was built from pandas' `DataFrame` class, which is why `titanic` holds its table and also knows how to `.describe()` it. Today you write classes yourself. We will grow one example, a `Student` with a name and a list of scores, one idea at a time: storing data on an object, writing methods, and what the small word `self` means. Later a `Course` will hold many students, and at the end you will see that a scikit-learn model is an object too. That last part prepares you for this week's assignment and for the final project, which asks every group to write at least two classes of its own.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Explain what a class, an object (also called an instance), an attribute, and a method are, with pandas objects you already know as examples.
- Write a class whose `__init__` method stores data on each new object, and create several independent objects from it.
- Write methods that return a value and methods that change the object, and know which kind gives back `None`.
- Explain what `self` is, and read the error messages you get when it is missing.
- Give a class a readable printout with `__repr__`, a value that all its objects share, and a list of other objects to manage, the way a course manages its students.
- Read a class that inherits from another class, such as `class GradStudent(Student):`.
- Use a scikit-learn model as an object (create it, `fit` it, `predict`, `score`), and write a small model class of your own with the same methods.
- Decide when a function is enough and when a class is worth writing.

### ✅ Prerequisites
- Session 10: parameters and default values, `return` and `None`, scope (a function's local variables disappear when it returns), `lambda` with `max(key=...)`, and list and dict comprehensions.
- Session 07: `sorted()` returns a new list, while `.sort()` changes the original. From the pandas sessions you need `value_counts()` and `groupby`.
- Assignment 06: the logistic regression you fit with `smf.logit`.
- Run the Setup cell to import the libraries and load the Titanic data.

### 🧩 Key Concepts
`object`  `class`  `instance`  `attribute`  `method`  `instantiation`  `__init__`  `self`  `state`  `__repr__`  `dunder method`  `class attribute`  `composition`  `inheritance`  `super()`  `fit / predict / score`

### 📖 Reading
- The official Python tutorial, [Chapter 9: Classes](https://docs.python.org/3/tutorial/classes.html), sections 9.3 and 9.5
- McKinney, *Python for Data Analysis*, §2.3, the short parts on objects, attributes and methods, and duck typing

### 🗺️ Today's Agenda
- 2.1 You already use objects
- 2.2 Why write a class of your own?
- 2.3 Your first class: `class` and `__init__`
- 2.4 Methods
- 2.5 What `self` really is
- 2.6 Objects remember
- 2.7 Readable objects: `__repr__` and class attributes
- 2.8 Objects that hold other objects
- 2.9 Inheritance, briefly
- 2.10 Models are objects
- 2.11 Function or class?

### 🧭 Two Ways to Organize Code
<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pp_vs_oop.png" width="800" alt="Procedural programming as a staircase of steps, next to object-oriented programming as one class with several objects built from it">
</p>

Figure source: Made with Claude. The staircase and car diagrams inside it are from Scaler Topics.

Everything you have written so far is procedural: a list of steps, each one a function, with data handed from one step to the next. Today adds a second style, in which data and the functions that work on it travel together as objects. Section 2.2 shows why that helps, with an analysis you already know how to run.

---
## ▶️ Setup — run this cell first
This imports the libraries and loads the Titanic passengers, who have been with us since the pandas sessions. Most of today's code is plain Python. We use pandas for a few tables, and scikit-learn for the logistic regression in section 2.10.

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression    # for section 2.10

# The Titanic passengers, read straight from the public course repo (no downloads).
TITANIC_URL = "https://raw.githubusercontent.com/cgpan/hudm5001/main/asset/titanic.csv"
titanic = pd.read_csv(TITANIC_URL)

print("pandas", pd.__version__, "| NumPy", np.__version__)
print("Setup complete ✅")

---
## 2. Content

### 2.1 You already use objects
Every value in Python is an object, and every object belongs to a class. The class decides what the object holds and what it can do. `type()` tells you an object's class.

In [ ]:
print(type("Teachers College"))    # a string
print(type([88, 92, 79]))          # a list
print(type(titanic))               # a DataFrame
print(type(titanic["fare"]))       # a Series

Each line prints the word `class`. A string, a list, and a DataFrame are all objects, built from the classes `str`, `list`, and `DataFrame`. For the pandas classes, Python also prints where the class lives inside the pandas package, so the class name is the last word.

An object carries two kinds of things, and you reach both with a dot. An attribute is a piece of data stored on the object, and you read it without parentheses. A method is a function that belongs to the object, and you call it with parentheses. You have used both since the first pandas session.

In [ ]:
print(titanic.shape)          # <---- attribute: data stored on the object, no parentheses
print(titanic.columns)        # <---- another attribute
print(titanic.head(3))        # <---- method: a function that belongs to the object, called with ()

name = "ann lee"
print(name.title())           # strings have methods too

Methods come in two kinds. Some hand back something new and leave the object as it was: `name.title()` returns a new string, and `name` stays lowercase. Others change the object itself and usually hand back `None`, like `.append()` and `.sort()` on a list. You met this difference in Session 07, where `sorted()` returned a new list while `.sort()` reordered the original.

In [ ]:
print(name.title(), "|", name)    # title() returned a new string; name is unchanged

scores = [88, 92, 79]
result = scores.append(95)        # append() changes the list itself
print(scores)                     # [88, 92, 79, 95]
print(result)                     # None, like a function without a return

# scores = scores.append(95)      # a classic bug: scores becomes None and the list is lost

A class is a blueprint for objects. I find it easiest to picture a blank form, like the record a school keeps for each student. The blank form decides which fields every record has, say a name and a list of scores, and what the office can do with a finished record, such as add a score or compute the average. Each filled-in form is one object, also called an instance of the class. The office can fill in as many as it needs from the same blank, and each one holds its own answers. In pandas, `DataFrame` is the blank form and `titanic` is one filled-in copy.

| Word | What it means | pandas example |
|---|---|---|
| class | the blueprint: a type of object | `pd.DataFrame` |
| object, or instance | one thing built from a class | `titanic` |
| attribute | data stored on an object, read without `()` | `titanic.shape` |
| method | a function that belongs to the class, called with `()` | `titanic.head()` |
| instantiation | creating an object by calling its class | `pd.DataFrame({"x": [1, 2]})` |

<!-- FIGURE SLOT 1, for the instructor. This comment does not show in the rendered notebook.
Suggested file name: class_vs_object.png
Suggested ChatGPT prompt: A clean, flat teaching illustration on a white background. On the left, one blank form with the title "class Student", two empty fields labeled "name" and "scores", and two buttons labeled "average()" and "add_score()". Three arrows lead to three filled-in copies of the form on the right, titled "ann", "bob", and "cara", each with its own name and list of scores and the same two buttons. Small caption under the left form: "class: the blank form (blueprint)". Small caption under the right: "objects (instances): filled-in copies". No other text.
When the image is in the repo's figures/ folder, replace this whole comment with:
<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/class_vs_object.png" width="800">
</p>

Figure source: Generated by ChatGPT.
-->

To see everything an object offers, type its name and a dot, such as `titanic.`, and press `Tab`. Colab and Jupyter list its attributes and methods. `dir(titanic)` prints the same list, and it is a long one.

#### ✏️ Try It Yourself — Exercise 1
**Difficulty:** ★☆☆☆☆  **Skills:** `type()`, attributes and methods

The next cell stores the `fare` column in `fares`.  
(a) Print the class of `fares` with `type()`.  
(b) Print the `type()` of `fares.size`, `fares.max`, `fares.shape`, and `fares.describe`, with no parentheses after them, as in `print(type(fares.max))`. A method shows up as `<class 'method'>`. Which two are methods, and which two are attributes?  
(c) Print the two attributes, and call the two methods with parentheses. Then try parentheses on an attribute, as in `fares.size()`, copy the error message into a comment, and delete the line.  
(d) What does `print(fares.max)` show, without parentheses? Describe it in a comment.

In [ ]:
fares = titanic["fare"]

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
fares = titanic["fare"]

# (a)
print(type(fares))         # a Series

# (b)
print(type(fares.size))            # <class 'int'>: an attribute
print(type(fares.max))             # <class 'method'>: a method
print(type(fares.shape))           # <class 'tuple'>: an attribute
print(type(fares.describe))        # <class 'method'>: a method

# (c)
print(fares.size, fares.shape)     # 891 (891,)
print(fares.max())                 # 512.33
print(fares.describe())            # a summary table
# fares.size()                     # TypeError: 'int' object is not callable. size is a number, and a number cannot be called

# (d)
print(fares.max)                   # a "bound method": the method itself, followed by the Series it belongs to
```

</details>

### 2.2 Why write a class of your own?
Start with an analysis you already know how to run. Every semester a new file of student records arrives, like the dropout data from Assignment 06, and you run the same five steps on it.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pipeline_example.png" width="800" alt="Five steps, load, clean, engineer features, fit a model and visualize, run on a new file every semester">
</p>

In the procedural style you know, each step is a function, and each arrow is a value that one step hands to the next.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pipeline_procedural.png" width="800" alt="The five steps as a staircase of function calls, each handing df or model to the next">
</p>

For one analysis, this is all you need.

Then the next semester arrives, and the next.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pipeline_pile_up.png" width="800" alt="Sticky notes with variable names piling up across semesters, such as df_fall, model_spring, df_final2 and model_new_REAL">
</p>

Look at what piles up. Some of it is data, and some of it is numbers that a step learned from the data: the medians `clean()` used to fill in missing values, and the mean and SD `engineer()` used to make z-scores. Suppose you want fall's model to flag at-risk students in spring. Spring then has to be cleaned and scaled with fall's numbers, so you keep those numbers in loose variables and pass them back into each function by hand. The functions cannot hold on to the numbers for you, because a function's local variables vanish when it returns.

The model's coefficients are the one thing you never had to carry. `model` kept them itself and used them when it predicted, because `model` is an object. (You will see them as `model.coef_` in 2.10.) What if the whole analysis worked like `model`?

Here is the same analysis written as a class. `Analysis` is the blueprint, and each semester becomes one object that keeps its own data, its own learned numbers, and its own model.

<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/pipeline_class_objects.png" width="800" alt="A class named Analysis with what it knows and what it can do, and three objects built from it, one per semester">
</p>

`Analysis` is a sketch, not code in this notebook. Starting in 2.3 we build a smaller class, `Student`, one idea at a time. The data class you write for your final project can look a lot like `Analysis`.

Here are the two styles side by side.

| | Procedural | Object-oriented |
|---|---|---|
| Built around | steps (functions) | things (objects) that hold data and act on it |
| Where the data lives | in loose variables, handed from step to step | inside the object, as attributes |
| Numbers a step learns | returned, then passed back in by hand | kept on the object, so each semester's object keeps its own |
| A new semester | rerun the calls and rename every variable | one more object, such as `Analysis("fall_2027.csv")` |
| Picture it as | a recipe | a blueprint and the things built from it |

Figure source for the four figures above: Made with Claude.

Now a small version you can run yourself. Suppose you keep a gradebook. Without classes, you might store each student as a dictionary and write a few functions that take that dictionary as their first argument.

In [ ]:
def make_student(name):
    """Return a new student record: a dictionary with a name and an empty list of scores."""
    return {"name": name, "scores": []}

def add_score(record, score):
    """Add one score to a student record."""
    record["scores"].append(score)

def average(record):
    """Return the mean score of a student record."""
    return sum(record["scores"]) / len(record["scores"])

ann = make_student("Ann")
add_score(ann, 90)
add_score(ann, 85)
print(ann)
print(average(ann))

This works, and in a short notebook it is fine. Two problems grow with the code. Nothing ties these functions to student records, so someone reading your notebook has to work out which functions go with which dictionaries, and `average` will run on any dictionary that happens to have a `"scores"` key. The names also start to collide. An average for a whole course would need another name, such as `course_average`, and soon every function name has to repeat the name of the thing it works on.

A class keeps the data and the functions that work on it in one place. Functions defined inside a class are its methods, and you reach them through the object with a dot, the way you call `titanic.head()`. Once we have written the `Student` class in 2.3 and 2.4, the gradebook code will read like this:

```python
ann = Student("Ann", [90, 85])    # create a student
ann.add_score(95)                 # the student updates itself
ann.average()                     # and computes its own average
```

A `Course` class can then have an `average()` method of its own, and the two never clash, because each one lives inside its own class.

### 2.3 Your first class: `class` and `__init__`
A class definition starts with the keyword `class`, the class name, and a colon. Class names use CapWords, as in `Student` or `DataFrame`, while functions and variables use lowercase words joined by underscores. Inside the class, a method named `__init__` sets up each new object. Its name has two underscores on each side, and Python calls it for you every time you create an object. The `print` line inside it below is only there so you can see when it runs.

In [ ]:
class Student:
    """A student with a name and a list of scores."""

    def __init__(self, name, scores):
        print(f"__init__ is running for {name}")    # only here to show when __init__ runs
        self.name = name          # <---- store the argument on the object: an attribute
        self.scores = scores      # <---- every Student object gets its own name and scores

ann = Student("Ann", [90, 85])          # <---- instantiation: call the class like a function
bob = Student("Bob", [72, 88, 95])

In [ ]:
print(ann.name, ann.scores)     # read attributes with a dot, as with titanic.shape
print(bob.name, bob.scores)
print(type(ann))                # Student is a new type of our own

Calling `Student("Ann", [90, 85])` does three things. Python creates a new, empty `Student` object. It calls `__init__` with that object as `self` and your two arguments as `name` and `scores`, and `__init__` stores them on the object. Then Python hands the finished object back, and we keep it in `ann`. You passed two arguments while `__init__` lists three parameters, because Python fills in `self` by itself. Section 2.5 shows how. (In `<class '__main__.Student'>`, the `__main__` part means the class was defined in this notebook.)

Syntax:  
```python
class ClassName:                                     # <---- the keyword class, a name in CapWords, a colon
    """One sentence on what an object of this class is."""

    def __init__(self, parameter_1, parameter_2):    # <---- runs every time you create an object
        self.attribute_1 = parameter_1               # <---- self.name = value stores data on the new object
        self.attribute_2 = parameter_2

new_object = ClassName(argument_1, argument_2)       # <---- create an object; leave self out of the call
```

Where do the attributes live? Each object keeps them in a dictionary of its own, and `vars()` shows it. The student record from 2.2 is still a dictionary underneath, now stored inside the object.

In [ ]:
print(vars(ann))     # the attributes stored on ann
print(vars(bob))     # bob has a dictionary of its own

### 2.4 Methods
A method is a function written inside a class. It looks like any `def` from last week, with one difference: its first parameter is `self`, the object the method was called on. Inside a method, `self.scores` means the scores of whichever student this is. Below, the class gets two methods, one of each kind from 2.1. `average()` returns a value and leaves the student alone, and `add_score()` changes the student and returns nothing.

In [ ]:
class Student:
    """A student with a name and a list of scores."""

    def __init__(self, name, scores):
        self.name = name
        self.scores = scores

    def average(self):                                  # <---- a method: a def inside the class, self first
        """Return the mean of this student's scores."""
        return sum(self.scores) / len(self.scores)      # <---- self.scores: this student's scores

    def add_score(self, score):
        """Add one score to this student's list. Returns nothing."""
        self.scores.append(score)                       # <---- changes the object itself

ann = Student("Ann", [90, 85])        # the class changed, so we create the students again
bob = Student("Bob", [72, 88, 95])

print(ann.average())     # 87.5
print(bob.average())     # 85.0

<!-- FIGURE SLOT 2, for the instructor. This comment does not show in the rendered notebook.
Suggested file name: class_anatomy.png
Suggested ChatGPT prompt: A labeled code diagram on a light background, in the style of a textbook figure. Show this Python code in a monospace font, exactly as written: "class Student:", "def __init__(self, name, scores):", "self.name = name", "self.scores = scores", "def average(self):", "return sum(self.scores) / len(self.scores)", with normal Python indentation. Add callout arrows with short labels: on "Student", "class name (CapWords)"; on "__init__", "runs when an object is created"; on "self", "the object being built or used"; on "self.name = name", "stores data on the object (an attribute)"; on "def average(self)", "a method: a function that belongs to the class". No other text.
When the image is in the repo's figures/ folder, replace this whole comment with:
<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/class_anatomy.png" width="800">
</p>

Figure source: Generated by ChatGPT.
-->

In [ ]:
ann.add_score(95)        # changes ann and returns None, so there is nothing to print
print(ann.scores)        # [90, 85, 95] (each extra run of this cell adds another 95)
print(ann.average())     # 90.0

> ⚠️ **Objects keep the class they were built from.** When you change a class and run its cell again, Python makes a new class with the same name. Objects you created earlier still belong to the old version, so they do not get the new methods. That is why the cell above creates `ann` and `bob` again. If Python says an object has no attribute that you just added to the class, rerun the cell that creates the object.

The two kinds of methods lead to a classic bug. Since `add_score` returns `None`, assigning its result back to the student throws the student away.

In [ ]:
cara = Student("Cara", [78, 82])
cara = cara.add_score(80)    # add_score changes cara, then returns None...
print(cara)                  # ...so the name cara now holds None, and the student is gone

`add_score` has already updated the student by the time it returns, so call it on a line of its own, with no `=`. The other common slip is leaving off the parentheses. You then get the method itself, which Python calls a bound method: a method tied to `ann` and waiting to be called. It is the same thing Exercise 1 showed with `fares.max`.

In [ ]:
print(ann.average)       # no parentheses: the method itself
print(ann.average())     # with parentheses: the method runs

#### ✏️ Try It Yourself — Exercise 2
**Difficulty:** ★★★☆☆  **Skills:** `class`, `__init__`, attributes, methods that return a value and methods that change the object

Last week's Exercise 5 stored four programs as tuples of a name and an enrollment. Write a class `Program` instead.  
(a) `__init__(self, name, enrolled, capacity=40)` stores the three values as attributes. A default value works in `__init__` the same way as in any function.  
(b) `seats_left()` returns the capacity minus the enrollment, and `is_full()` returns `True` when no seats are left. Inside a method, reach the attributes through `self` (`self.capacity`), and call another method through `self` with parentheses (`self.seats_left()`).  
(c) `enroll(n)` adds `n` students to the enrollment and returns nothing.  
(d) Create `Program("Measurement", 24)` and print its seats left. Enroll 16 students, then print `seats_left()` and `is_full()` again.  
(e) Build a list of `Program` objects from `programs` with a list comprehension, and print each program's name with its seats left.

In [ ]:
programs = [("Applied Stats", 32), ("Learning Analytics", 28), ("Measurement", 24), ("Cognitive Science", 36)]

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
class Program:
    """A degree program with an enrollment and a capacity."""

    def __init__(self, name, enrolled, capacity=40):     # (a)
        self.name = name
        self.enrolled = enrolled
        self.capacity = capacity

    def seats_left(self):                                 # (b)
        """Return how many seats are still open."""
        return self.capacity - self.enrolled

    def is_full(self):
        """Return True when no seats are left."""
        return self.seats_left() <= 0                     # one method calling another through self

    def enroll(self, n):                                  # (c)
        """Add n students to the program. Returns nothing."""
        self.enrolled += n

# (d)
m = Program("Measurement", 24)
print(m.seats_left())                  # 16
m.enroll(16)
print(m.seats_left(), m.is_full())     # 0 True

# (e)
all_programs = [Program(name, n) for name, n in programs]
for p in all_programs:
    print(p.name, "has", p.seats_left(), "seats left")
```

</details>

### 2.5 What `self` really is
`self` confuses nearly everyone at first, so let me show you what Python does with it. When you write `ann.average()`, Python looks up `average` in the class of `ann`, which is `Student`, and calls it with `ann` as the first argument. The two lines below do the same thing.

In [ ]:
print(ann.average())             # the usual way to call a method
print(Student.average(ann))      # what Python runs for you: ann goes in as self

So `self` is the object in front of the dot. While `ann.average()` runs, `self` is `ann`, and `self.scores` is `ann.scores`. While `bob.average()` runs, `self` is `bob`. The class holds one copy of each method, and every student shares it, because each call passes in a different object. In your own code you will always use the short form. The long form is here only to show where `self` comes from.

Compare this with 2.2, where the function `average(record)` took the student record as its first argument. The method `average(self)` is the same function moved inside the class, with `self` in the place of `record`, and `Student.average(ann)` looks just like the old call `average(ann)`.

<!-- FIGURE SLOT 3, for the instructor. This comment does not show in the rendered notebook.
Suggested file name: self_method_call.png
Suggested ChatGPT prompt: A simple teaching diagram on a white background with two lines of monospace code. Top: "ann.average()". A curved arrow labeled "the same call" leads to the bottom line: "Student.average(ann)". Below that, the definition "def average(self):", with an arrow from "ann" in the bottom line to "self", labeled "the object in front of the dot becomes self". Flat colors, minimal text, and the code text exactly as written.
When the image is in the repo's figures/ folder, replace this whole comment with:
<p align="left">
  <img src="https://raw.githubusercontent.com/cgpan/hudm5001/main/figures/self_method_call.png" width="800">
</p>

Figure source: Generated by ChatGPT.
-->

This also explains two error messages you are going to meet. In the first class below, I forgot `self` in the method definition.

In [ ]:
class Broken:
    """A class with a bug: its method has no self parameter."""

    def hello():                       # <---- forgot self
        return "hello"

b = Broken()
# b.hello()                            # TypeError: Broken.hello() takes 0 positional arguments but 1 was given

Delete the `#` in front of `b.hello()` and run the cell. You called `hello` with nothing in the parentheses, yet Python says 1 argument was given. That argument is `b`, which Python passed in as the first argument, and `hello` has no parameter to receive it. Put the `#` back afterwards so the notebook still runs from top to bottom.

The second mistake is forgetting `self.` when you store a value.

In [ ]:
class Forgetful:
    """A class with a bug: __init__ never stores the name on the object."""

    def __init__(self, name):
        name = name                    # <---- forgot self.: this only assigns the local variable to itself

    def greet(self):
        return f"Hi, {self.name}!"

f = Forgetful("Ann")
print(vars(f))                         # {}: nothing was stored on the object
# f.greet()                            # AttributeError: 'Forgetful' object has no attribute 'name'

This is last week's scope rule at work. Inside `__init__`, the parameter `name` is a local variable, so `name = name` only assigns it to itself. Like every local variable, it disappears when `__init__` returns. `self.name = name` stores the value on the object, which lives on after `__init__` has finished.

One last fact: `self` is only the usual name. Python passes the object as the first argument whatever you call that parameter, just as `*args` could have been `*numbers` last week. Every Python programmer writes `self`, though, and I recommend that you do too.

#### ✏️ Try It Yourself — Exercise 3
**Difficulty:** ★★★☆☆  **Skills:** `self`, reading error messages

The class `Book` in the next cell has three bugs, and each one has to do with `self`. Remove the `#` in front of the first test line and run the cell. Read the error, fix the class, and run the cell again. Then do the same with the second test line. One fix can uncover the next bug, so keep running the cell after each change. Next to each fix, write a comment with the error message you saw and what caused it. When the class works, the two test lines print `880` and `True`.

In [ ]:
class Book:
    """A book with a title and a number of pages."""

    def __init__(self, title, pages):
        self.title = title
        pages = pages

    def is_long():
        """Return True if the book has more than 300 pages."""
        return pages > 300

novel = Book("Middlemarch", 880)
# print(novel.pages)        # test 1
# print(novel.is_long())    # test 2

<details>
<summary>▶ Show solution</summary>

```python
class Book:
    """A book with a title and a number of pages."""

    def __init__(self, title, pages):
        self.title = title
        self.pages = pages      # fix 1. AttributeError: 'Book' object has no attribute 'pages'
                                # pages = pages only assigned the local variable to itself, and it vanished when __init__ ended

    def is_long(self):          # fix 2. TypeError: Book.is_long() takes 0 positional arguments but 1 was given
                                # Python passes novel in as the first argument, so the method needs self
        """Return True if the book has more than 300 pages."""
        return self.pages > 300 # fix 3. NameError: name 'pages' is not defined
                                # inside a method, the object's data is reached through self

novel = Book("Middlemarch", 880)
print(novel.pages)          # test 1: 880
print(novel.is_long())      # test 2: True
```

</details>

### 2.6 Objects remember
An object keeps its attributes for as long as the object exists, so each method call can build on the calls before it. Each student also keeps a separate list of scores.

In [ ]:
ann = Student("Ann", [90, 85])
bob = Student("Bob", [72, 88, 95])

ann.add_score(95)
ann.add_score(80)
print(ann.scores, ann.average())    # [90, 85, 95, 80] 87.5: each call built on the last
print(bob.scores, bob.average())    # [72, 88, 95] 85.0: bob's list was not touched

The values an object holds at a given moment are called its state. `add_score` changes the state and `average` reads it. Last week's functions kept nothing between calls. Their local variables vanished when they returned, so any value that had to last lived outside the function, like `count` in `count = bumped(count)` or the dictionaries in 2.2. An object carries its state with it, together with the methods that read and change it, and that pairing is the main reason to write a class. A fitted model in 2.10 works exactly this way. `fit` stores what the model learned, and `predict` uses it later.

What about a new student with no scores yet? A default value is the natural fix, and the natural default looks like an empty list. Before you run the next cell, write down what you expect the two `print` lines to show.

In [ ]:
class Student:
    """A student with a name and a list of scores. This version has a bug."""

    def __init__(self, name, scores=[]):     # <---- an empty list as the default: the trap
        self.name = name
        self.scores = scores

    def add_score(self, score):
        """Add one score to this student's list. Returns nothing."""
        self.scores.append(score)

dan = Student("Dan")
eve = Student("Eve")
dan.add_score(77)
print("Dan:", dan.scores)
print("Eve:", eve.scores)

Eve shows Dan's score. Python creates a default value only once, when it runs the `def` line. Every student created without scores gets that same list, so Dan and Eve share one list, and a score added for one shows up for the other.

The fix is to use `None` as the default and create a new list inside `__init__`. The conditional expression from Session 07 does it in one line. I also made `average()` return `None` when there are no scores yet, since dividing by zero would crash. It uses an early `return`, like `fare_band` last week.

In [ ]:
class Student:
    """A student with a name and a list of scores."""

    def __init__(self, name, scores=None):
        self.name = name
        self.scores = [] if scores is None else scores    # <---- a new empty list for each student

    def average(self):
        """Return the mean of this student's scores, or None if there are none yet."""
        if len(self.scores) == 0:
            return None
        return sum(self.scores) / len(self.scores)

    def add_score(self, score):
        """Add one score to this student's list. Returns nothing."""
        self.scores.append(score)

dan = Student("Dan")
eve = Student("Eve")
dan.add_score(77)
print("Dan:", dan.scores, dan.average())
print("Eve:", eve.scores, eve.average())    # a list of its own, still empty

> ⚠️ **Do not use a list or a dictionary as a default value.** Use `None`, and create the list inside the function. The trap is the same in any function with a default value.

#### ✏️ Try It Yourself — Exercise 4
**Difficulty:** ★★★☆☆  **Skills:** state that builds up across calls, `None` as a default, independent objects

Write a class `Survey` for one survey question.  
(a) `__init__(self, question, responses=None)` stores the question and a list of responses. Use `None` as the default, as in the fixed `Student`.  
(b) `add(answer)` stores one more response and returns nothing. `count(answer)` returns how many times that answer was given (use the list's own method, `self.responses.count(answer)`), and `n()` returns the number of responses so far.  
(c) Create a survey for the question in `question`, add the answers in `answers` one at a time with a `for` loop, and print `count("agree")` and `n()`.  
(d) Create a second `Survey` with a question of your own and print its `n()`. It should be `0`. If it is not, look at your default value again.

In [ ]:
question = "The readings helped me understand the lecture."
answers = ["agree", "neutral", "agree", "strongly agree", "disagree", "agree"]

# Your code here

<details>
<summary>▶ Show solution</summary>

```python
class Survey:
    """Responses to one survey question."""

    def __init__(self, question, responses=None):          # (a)
        self.question = question
        self.responses = [] if responses is None else responses

    def add(self, answer):                                  # (b)
        """Store one more response. Returns nothing."""
        self.responses.append(answer)

    def count(self, answer):
        """Return how many times answer was given."""
        return self.responses.count(answer)

    def n(self):
        """Return the number of responses so far."""
        return len(self.responses)

# (c)
readings = Survey(question)
for a in answers:
    readings.add(a)
print(readings.count("agree"), "of", readings.n())    # 3 of 6

# (d)
pace = Survey("The pace of the class was about right.")
print(pace.n())                                       # 0: a new list of its own
```

</details>

### 2.7 Readable objects: `__repr__` and class attributes
Print a student and Python shows the class name and a memory address, which tells you nothing about the student.

In [ ]:
print(ann)

To decide what to show, Python calls a special method named `__repr__` (short for representation). Define it in your class and return a string, and `print()` and the notebook will both show that string. A common habit is to make it look like the code that would create the object.

Methods whose names start and end with two underscores are called dunder methods, for double underscore. You rarely call them yourself. Python calls them at the right moment: `__init__` when an object is created, `__repr__` when it is shown, and `__len__` when you call `len()` on it. That is why `len(titanic)` gives you the number of rows. The `DataFrame` class defines `__len__`, and its `__repr__` draws the table you see when you print a DataFrame. (Tutorials often use `__str__` as well. When a class has one, `print()`, `str()`, and f-strings use it, and they fall back on `__repr__` when it does not. If you write just one, write `__repr__`.)

The new version of `Student` below also has a class attribute and a method that calls another method.

In [ ]:
class Student:
    """A student with a name and a list of scores."""

    passing_score = 60                         # <---- class attribute: one value shared by every student

    def __init__(self, name, scores=None):
        self.name = name                       # <---- instance attributes: each student has their own
        self.scores = [] if scores is None else scores

    def average(self):
        """Return the mean of this student's scores, or None if there are none yet."""
        if len(self.scores) == 0:
            return None
        return sum(self.scores) / len(self.scores)

    def add_score(self, score):
        """Add one score to this student's list. Returns nothing."""
        self.scores.append(score)

    def passed(self):
        """Return True if the average reaches the passing score, and False with no scores yet."""
        if self.average() is None:             # <---- a method can call another method through self
            return False
        return self.average() >= self.passing_score

    def __repr__(self):                        # <---- dunder method: Python calls it to show the object
        """Return a short description, written like the code that creates the student."""
        return f"Student(name='{self.name}', scores={self.scores})"

ann = Student("Ann", [90, 85, 95])
bob = Student("Bob", [52, 61, 58])
print(ann)                  # print() calls ann.__repr__() for you
print([ann, bob])           # a list shows each student through __repr__ as well
print(bob.passed())         # False: an average of 57 is below 60

A class attribute sits directly under the `class` line, outside every method. It belongs to the class itself, so all objects share one value. Use it for something that is the same for every object, like a course-wide passing score. Values that differ from object to object, like a name or a list of scores, go in `__init__` as `self.something`. A class attribute is not stored on each object, so `vars(ann)` does not list it. When an object has no attribute of its own with that name, Python looks in the class.

In [ ]:
print(Student.passing_score)       # read it from the class
print(ann.passing_score)           # or from any student: Python finds it in the class
print(vars(ann))                   # passing_score is not stored on ann itself

Student.passing_score = 55         # change the rule once, for every student
print(bob.passed())                # True: 57 passes now
Student.passing_score = 60         # put the rule back

# ann.passing_score = 55           # this would store a separate 55 on ann alone; everyone else keeps 60

#### ✏️ Try It Yourself — Exercise 5
**Difficulty:** ★★☆☆☆  **Skills:** `__repr__`, class attributes

Copy your `Program` class from Exercise 2 (or the one in its solution) into the next cell and extend it.  
(a) Add a `__repr__` method that returns a string such as `Program(name='Measurement', enrolled=24, capacity=40)`. It has to `return` the string. If it prints the string instead, `print()` fails with a `TypeError` about `__str__`.  
(b) Add a class attribute `school = "Teachers College"`. Print it from the class, and from one program.  
(c) Build the list of programs again with your comprehension from Exercise 2, and print the whole list.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
class Program:
    """A degree program with an enrollment and a capacity."""

    school = "Teachers College"                           # (b) shared by every program

    def __init__(self, name, enrolled, capacity=40):
        self.name = name
        self.enrolled = enrolled
        self.capacity = capacity

    def seats_left(self):
        """Return how many seats are still open."""
        return self.capacity - self.enrolled

    def is_full(self):
        """Return True when no seats are left."""
        return self.seats_left() <= 0

    def enroll(self, n):
        """Add n students to the program. Returns nothing."""
        self.enrolled += n

    def __repr__(self):                                   # (a)
        """Return a short description, written like the code that creates the program."""
        return f"Program(name='{self.name}', enrolled={self.enrolled}, capacity={self.capacity})"

# (b)
print(Program.school)
print(Program("Measurement", 24).school)

# (c)
all_programs = [Program(name, n) for name, n in programs]
print(all_programs)
```

</details>

### 2.8 Objects that hold other objects
Real programs use several classes together. A course has many students, so a `Course` object can keep a list of `Student` objects and offer methods that work across all of them. Programmers call this composition: an object built from other objects. Most of the methods below reuse a tool from last week.

In [ ]:
class Course:
    """A course with a title and a list of Student objects."""

    def __init__(self, title):
        self.title = title
        self.students = []                     # <---- will hold Student objects

    def add_student(self, student):
        """Add one Student object to the course. Returns nothing."""
        self.students.append(student)

    def average(self):
        """Return the mean of the students' averages."""
        averages = [s.average() for s in self.students]          # a list comprehension over the students
        return sum(averages) / len(averages)

    def top_student(self):
        """Return the Student with the highest average."""
        return max(self.students, key=lambda s: s.average())    # max with a lambda key, as in Session 10

    def to_dataframe(self):
        """Return a DataFrame with one row per student: name, number of scores, and average."""
        rows = [{"name": s.name, "n_scores": len(s.scores), "average": s.average()} for s in self.students]
        return pd.DataFrame(rows)

    def __len__(self):                         # <---- lets len(course) work, like len(titanic)
        """Return the number of students."""
        return len(self.students)

    def __repr__(self):
        """Return a short description of the course."""
        return f"Course(title='{self.title}', {len(self)} students)"

In [ ]:
hudm = Course("HUDM 5001")
hudm.add_student(Student("Ann", [90, 85, 95]))
hudm.add_student(Student("Bob", [52, 61, 58]))
hudm.add_student(Student("Cara", [78, 82, 91]))
hudm.add_student(Student("Dan", [88, 91, 79, 94]))

print(hudm)                      # Course.__repr__
print(len(hudm))                 # Course.__len__
print(round(hudm.average(), 2))  # Course.average: the mean of the four students' averages
print(hudm.top_student())        # a whole Student object, shown by Student.__repr__

Look at who does what. A `Student` knows its own scores and how to average them. The `Course` never computes a student's average itself. It asks each student with `s.average()`, the way a bigger function called small helpers last week. And as 2.2 promised, the two `average` methods live side by side: `hudm.average()` runs the course version, and `ann.average()` runs the student version.

`to_dataframe()` hands the results to pandas. `pd.DataFrame` accepts a list of dictionaries and turns each dictionary into a row, with the keys as column names.

In [ ]:
table = hudm.to_dataframe()
print(table.sort_values("average", ascending=False))

#### ✏️ Try It Yourself — Exercise 6
**Difficulty:** ★★★☆☆  **Skills:** a class that holds other objects, list and dict comprehensions

The next cell repeats the `Course` class with two new methods for you to finish, followed by the lines that rebuild `hudm`, since an object keeps the class it was built from. Replace the two `...` placeholders.  
(a) `passing_students()` returns a list with the names of the students who passed. Use each student's `passed()` method in a list comprehension with a filter.  
(b) `averages(ndigits=1)` returns a dictionary that maps each student's name to their average, rounded to `ndigits`, built with a dict comprehension.  
(c) Remove the `#` from the two test lines and run the cell. Then try `hudm.averages(2)` as well.

In [ ]:
class Course:
    """A course with a title and a list of Student objects."""

    def __init__(self, title):
        self.title = title
        self.students = []

    def add_student(self, student):
        """Add one Student object to the course. Returns nothing."""
        self.students.append(student)

    def average(self):
        """Return the mean of the students' averages."""
        averages = [s.average() for s in self.students]
        return sum(averages) / len(averages)

    def top_student(self):
        """Return the Student with the highest average."""
        return max(self.students, key=lambda s: s.average())

    def to_dataframe(self):
        """Return a DataFrame with one row per student: name, number of scores, and average."""
        rows = [{"name": s.name, "n_scores": len(s.scores), "average": s.average()} for s in self.students]
        return pd.DataFrame(rows)

    def passing_students(self):
        """Return the names of the students who passed."""
        return ...                             # (a) a list comprehension with a filter

    def averages(self, ndigits=1):
        """Return a dictionary {name: average rounded to ndigits} for every student."""
        return ...                             # (b) a dict comprehension

    def __len__(self):
        """Return the number of students."""
        return len(self.students)

    def __repr__(self):
        """Return a short description of the course."""
        return f"Course(title='{self.title}', {len(self)} students)"

# rebuild the course from the class above
hudm = Course("HUDM 5001")
for student_name, student_scores in [("Ann", [90, 85, 95]), ("Bob", [52, 61, 58]), ("Cara", [78, 82, 91]), ("Dan", [88, 91, 79, 94])]:
    hudm.add_student(Student(student_name, student_scores))

# print(hudm.passing_students())    # ['Ann', 'Cara', 'Dan']
# print(hudm.averages())            # {'Ann': 90.0, 'Bob': 57.0, 'Cara': 83.7, 'Dan': 88.0}

<details>
<summary>▶ Show solution</summary>

```python
class Course:
    """A course with a title and a list of Student objects."""

    def __init__(self, title):
        self.title = title
        self.students = []

    def add_student(self, student):
        """Add one Student object to the course. Returns nothing."""
        self.students.append(student)

    def average(self):
        """Return the mean of the students' averages."""
        averages = [s.average() for s in self.students]
        return sum(averages) / len(averages)

    def top_student(self):
        """Return the Student with the highest average."""
        return max(self.students, key=lambda s: s.average())

    def to_dataframe(self):
        """Return a DataFrame with one row per student: name, number of scores, and average."""
        rows = [{"name": s.name, "n_scores": len(s.scores), "average": s.average()} for s in self.students]
        return pd.DataFrame(rows)

    def passing_students(self):                            # (a)
        """Return the names of the students who passed."""
        return [s.name for s in self.students if s.passed()]

    def averages(self, ndigits=1):                         # (b)
        """Return a dictionary {name: average rounded to ndigits} for every student."""
        return {s.name: round(s.average(), ndigits) for s in self.students}

    def __len__(self):
        """Return the number of students."""
        return len(self.students)

    def __repr__(self):
        """Return a short description of the course."""
        return f"Course(title='{self.title}', {len(self)} students)"

# rebuild the course from the class above
hudm = Course("HUDM 5001")
for student_name, student_scores in [("Ann", [90, 85, 95]), ("Bob", [52, 61, 58]), ("Cara", [78, 82, 91]), ("Dan", [88, 91, 79, 94])]:
    hudm.add_student(Student(student_name, student_scores))

print(hudm.passing_students())    # ['Ann', 'Cara', 'Dan']
print(hudm.averages())            # {'Ann': 90.0, 'Bob': 57.0, 'Cara': 83.7, 'Dan': 88.0}
print(hudm.averages(2))           # Cara's average now shows as 83.67
```

</details>

### 2.9 Inheritance, briefly
If time runs short in class, read this section at home. Nothing later in the notebook depends on it.

Sometimes you need a special version of an existing class. In this example a graduate student is a student with everything a `Student` has, plus an advisor and a higher passing score. Instead of copying `Student`, you can write a class that inherits from it. Put the parent class in parentheses after the new class's name.

In [ ]:
class GradStudent(Student):                    # <---- GradStudent inherits everything from Student
    """A graduate student: a Student with an advisor and a higher passing score."""

    passing_score = 70                         # <---- overrides Student's value, for grad students only

    def __init__(self, name, advisor, scores=None):
        super().__init__(name, scores)         # <---- run Student's __init__ to store the name and scores
        self.advisor = advisor                 # <---- then add what is new

    def __repr__(self):                        # <---- overrides Student's __repr__
        """Return a short description, written like the code that creates the student."""
        return f"GradStudent(name='{self.name}', advisor='{self.advisor}', scores={self.scores})"

gina = GradStudent("Gina", "Dr. Lee", [72, 64, 68])
print(gina)
print(gina.average(), gina.passed())    # both methods come from Student; passed() uses 70 here
print(isinstance(gina, Student))        # True: a GradStudent is also a Student
print(isinstance(ann, GradStudent))     # False: a plain Student is not a GradStudent

Three new pieces are at work. First, `class GradStudent(Student):` makes `Student` the parent class, so a `GradStudent` gets every method of `Student` without retyping it: `average()`, `add_score()`, and `passed()` all work on `gina`. Second, anything the child defines under the same name replaces the parent's version, for the child only. This is called overriding. Gina's average of 68 would pass under the student rule of 60, but a `GradStudent` needs 70. Third, `super().__init__(name, scores)` calls the parent's `__init__`, so the child does not repeat how a student stores a name and scores. Any code written for a `Student`, such as our `Course`, also works with a `GradStudent`.

Of today's tools, inheritance is the one I expect you to use least in your own projects. A class that holds other objects, like `Course`, is usually simpler to write and to read. You will mostly meet inheritance in libraries. Every scikit-learn model inherits from a few shared base classes, and a neural network in PyTorch, a popular deep learning library, is written as a class that inherits from PyTorch's `nn.Module` and calls `super().__init__()` at the start of its own `__init__`.

### 2.10 Models are objects
Every model in scikit-learn is a class, and all of them share the same few method names. Here is the whole life of one model object: a logistic regression that predicts Titanic survival from sex, passenger class, and fare. By scikit-learn convention, `X` (capital) holds the predictors and `y` holds the outcome.

In [ ]:
X = pd.DataFrame({
    "female": (titanic["sex"] == "female").astype(int),    # 1 for women, 0 for men
    "pclass": titanic["pclass"],
    "fare": titanic["fare"],
})
y = titanic["survived"]
print(X.head(3))

In [ ]:
model = LogisticRegression(max_iter=1000)    # <---- create the object: settings go into __init__
print(model)                                 # its __repr__ shows the settings you changed
# print(model.coef_)                         # AttributeError: 'LogisticRegression' object has no attribute 'coef_'

model.fit(X, y)                              # <---- fit() learns from the data and stores the results on the object
coefs = pd.Series(model.coef_[0], index=X.columns)    # <---- coef_ ends with an underscore: learned from the data
print(coefs.round(3))
print(model.intercept_.round(3))

print(model.predict(X.head(3)))              # <---- predict() uses what fit() stored
print(round(model.score(X, y), 3))           # <---- score() returns the share of correct predictions

Read the cell as the story of one object. `LogisticRegression(max_iter=1000)` runs `__init__`, which only stores your settings, because the model has not seen any data yet. `fit()` changes the object. It estimates the coefficients and stores them as attributes, and scikit-learn marks every attribute learned from data with a trailing underscore, as in `coef_` and `intercept_`. Those attributes do not exist until `fit()` has run. `predict()` and `score()` then read the stored numbers, the same way `average()` reads `self.scores`. `coef_` holds one row with a coefficient for each column of `X`, so the cell puts that row, `coef_[0]`, into a Series labeled with the column names. The signs read as in Assignment 06: women were more likely to survive than men, and passengers with a higher class number, meaning a cheaper ticket, were less likely to survive.

scikit-learn's `fit()` is an exception to the rule from 2.1. It changes the model and also hands the model back, so a cell that ends with `model.fit(X, y)` displays the model, and `LogisticRegression().fit(X, y).score(X, y)` works in one line. statsmodels, which you used in Assignment 06, does it differently. There, `smf.logit(formula, data=train)` created a model object, and its `.fit()` handed back a new results object, the one with `.params`, `.summary()`, and `.predict()`. That is why the worked example kept it in a variable: `demo_model = smf.logit(formula, data=train).fit()`.

Now that you know what `fit()` does inside, you can write a model of your own. The simplest one ignores the predictors and predicts the most common outcome for everyone. It makes a good baseline, since any real model should beat it.

In [ ]:
class MajorityClassifier:
    """Predict the most common outcome for everyone, ignoring the predictors."""

    def fit(self, X, y):
        """Learn the most common outcome in y and store it on the object."""
        self.majority_ = y.value_counts().idxmax()    # <---- the most common value; learned, so the name ends with _

    def predict(self, X):
        """Return the stored outcome once for every row of X."""
        return np.full(len(X), self.majority_)        # an array with len(X) copies of the outcome

    def score(self, X, y):
        """Return the share of rows where the prediction matches y."""
        return (self.predict(X) == y).mean()          # a method calling another method

    def __repr__(self):
        """Return a short description, in the scikit-learn style."""
        return "MajorityClassifier()"

baseline = MajorityClassifier()
baseline.fit(X, y)
print(baseline.majority_)           # 0: most passengers did not survive
print(round(baseline.score(X, y), 3))

`MajorityClassifier` has no `__init__` because it has no settings. Python then uses a built-in setup that does nothing, so `MajorityClassifier()` takes no arguments. Its `fit()` follows the usual rule and returns `None`, so call it on a line of its own, as the loop below does.

Both classes have methods named `fit` and `score`, so one loop can handle both models. The loop works with any object that has those two methods, whatever its class. Python programmers call this duck typing, after the saying that if it walks like a duck and quacks like a duck, it is a duck.

In [ ]:
models = [MajorityClassifier(), LogisticRegression(max_iter=1000)]

for m in models:
    m.fit(X, y)
    print(m, round(m.score(X, y), 3))

This loop is the setup for this week's assignment. scikit-learn's neural network, `MLPClassifier`, has the same `fit`, `predict`, and `score` methods, so it can join the list without any change to the loop. One caution: we scored each model on the same passengers it learned from. In the assignment you will score on a separate test file, as in Assignment 06.

#### ✏️ Try It Yourself — Exercise 7
**Difficulty:** ★★★★★  **Skills:** a model class with a setting and a learned attribute, `groupby`, dict comprehension, `.map()`

Write `GroupMajorityClassifier`, a model that predicts the most common outcome within each group of one column. Grouped by `female`, it learns "women survived, men did not". The next cell has the skeleton; replace the three `...` placeholders.  
(a) `__init__(self, column)` stores the name of the column to group by. It is a setting, so it goes in `__init__` and has no trailing underscore.  
(b) `fit(X, y)` already computes `rates`, the survival rate of each group. Turn it into a dictionary `self.rule_` that maps each group to `1` if its rate is at least 0.5 and to `0` otherwise. A dict comprehension over `rates.items()` does this, as in Session 10, Exercise 7.  
(c) `predict(X)` looks up each row's group in `self.rule_`. Calling `.map(some_dict)` on a Series does this lookup for every value.  
(d) Add `GroupMajorityClassifier("female")` and `GroupMajorityClassifier("pclass")` to the list of models, run the loop again, and print the `rule_` of each new model. How does the one-column rule on `female` compare with the logistic regression?

In [ ]:
class GroupMajorityClassifier:
    """Predict the most common outcome within each group of one column."""

    def __init__(self, column):
        ...                                     # (a) store the column name on the object

    def fit(self, X, y):
        """Learn the most common outcome in each group of X[column]."""
        rates = y.groupby(X[self.column]).mean()    # the survival rate of each group
        self.rule_ = ...                        # (b) a dictionary {group: 1 or 0}

    def predict(self, X):
        """Return the learned outcome for each row's group."""
        return ...                              # (c)

    def score(self, X, y):
        """Return the share of rows where the prediction matches y."""
        return (self.predict(X) == y).mean()

    def __repr__(self):
        """Return a short description, in the scikit-learn style."""
        return f"GroupMajorityClassifier(column='{self.column}')"

# (d) your loop here

<details>
<summary>▶ Show solution</summary>

```python
class GroupMajorityClassifier:
    """Predict the most common outcome within each group of one column."""

    def __init__(self, column):
        self.column = column                    # (a) a setting

    def fit(self, X, y):
        """Learn the most common outcome in each group of X[column]."""
        rates = y.groupby(X[self.column]).mean()
        self.rule_ = {group: 1 if rate >= 0.5 else 0 for group, rate in rates.items()}    # (b)

    def predict(self, X):
        """Return the learned outcome for each row's group."""
        return X[self.column].map(self.rule_)   # (c)

    def score(self, X, y):
        """Return the share of rows where the prediction matches y."""
        return (self.predict(X) == y).mean()

    def __repr__(self):
        """Return a short description, in the scikit-learn style."""
        return f"GroupMajorityClassifier(column='{self.column}')"

# (d)
models = [MajorityClassifier(), LogisticRegression(max_iter=1000),
          GroupMajorityClassifier("female"), GroupMajorityClassifier("pclass")]
for m in models:
    m.fit(X, y)
    print(m, round(m.score(X, y), 3))

print(models[2].rule_)    # {0: 0, 1: 1}: men 0, women 1
print(models[3].rule_)    # {1: 1, 2: 0, 3: 0}: only first class is predicted to survive

# The rule on female scores 0.787, the same as the logistic regression. The two models disagree
# on only 4 of the 891 passengers, so sex, class, and fare together barely change the guesses.
```

</details>

### 2.11 Function or class?
You now have two ways to package code. Here is how I would choose.

| Your situation | Reach for | Why |
|---|---|---|
| turn some inputs into one output, such as a label or a summary number | a function | nothing needs to be remembered after it returns |
| several functions keep taking the same data as their first argument | a class, with those functions as methods | the data and the code that uses it stay together |
| something has to remember values between calls, such as a fitted model or a gradebook | a class | attributes on `self` last as long as the object |
| many things share one structure, such as students, programs, or models | a class, with one object for each | write the class once and create as many objects as you need |
| a one-off calculation in a notebook cell | no `def` and no class | a plain line of pandas is the easiest to read |

For the final project, each group writes at least two classes with at least one method each, and the report describes them and shows a sample run. They can be as simple as today's classes. Two shapes fit most projects: a class that loads and cleans your dataset, with methods such as `clean()` and `summary()`, and a class that fits a model or runs a test and reports the result, with methods such as `fit()` and `report()`. A class like `Course`, which holds your records and turns them into a DataFrame, works too.

---
## 3. Conclusions & Key Takeaways
- Every value in Python is an object built from a class. An attribute is data stored on an object, read without parentheses. A method is a function that belongs to the object, called with parentheses.
- `class Name:` defines a new type. Calling the class, as in `Student("Ann", [90, 85])`, creates an object (an instance), and Python runs `__init__`, which stores the object's data with `self.attribute = value`.
- A method is a `def` inside a class whose first parameter is `self`. `ann.average()` is shorthand for `Student.average(ann)`, so `self` is the object in front of the dot.
- If you forget `self` in a method definition, Python says the method "takes 0 positional arguments but 1 was given". If you forget `self.` when storing a value, the value becomes a local variable and disappears when the method returns.
- Some methods return a new value and leave the object alone. Others change the object and usually return `None`, so never write `x = x.append(...)`.
- An object keeps its state between calls. That makes a class the natural home for data that several functions share, and for a model that learns in `fit` and predicts later.
- Do not use a list or a dictionary as a default value. Use `None`, and create a new list inside `__init__`.
- Python calls dunder methods for you: `__init__` when an object is created, `__repr__` when it is shown, and `__len__` for `len()`. A class attribute holds one value that every object shares.
- A class can hold objects of another class, as a `Course` holds `Student`s. A class can also inherit from another with `class Child(Parent):`, reuse the parent's methods, and override the ones that should differ. `super().__init__()` runs the parent's setup.
- scikit-learn models are objects with the same method names. Settings go into `__init__`, `fit` stores learned attributes whose names end in `_` (and hands the model back), and `predict` and `score` use them. Any class with those method names, like our `MajorityClassifier`, can join the same loop.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| check an object's class | `type(x)`, `isinstance(x, Student)` |
| define a class | `class Student:` |
| set up each new object | `def __init__(self, name, scores=None):` then `self.name = name` |
| create an object | `ann = Student("Ann", [90, 85])` |
| read an attribute | `ann.name` |
| call a method | `ann.average()` |
| a method that changes the object | `def add_score(self, score): self.scores.append(score)` |
| see what is stored on an object | `vars(ann)` |
| readable printing | `def __repr__(self): return f"Student(name='{self.name}')"` |
| make `len()` work | `def __len__(self): return len(self.students)` |
| class attribute | `passing_score = 60` under the `class` line, read as `self.passing_score` |
| inherit and extend | `class GradStudent(Student):` with `super().__init__(name, scores)` |
| scikit-learn model | `m = LogisticRegression()`, `m.fit(X, y)`, `m.predict(X)`, `m.score(X, y)` |
| learned attributes | `m.coef_`, `m.intercept_` (they exist only after `fit`) |

### ⏭️ Coming Up Next
**Unit testing and exception handling.** Today's `Student` accepts whatever you give it. `ann.add_score("ninety")` goes into the list without a complaint, and `average()` crashes later. Next week you will learn to stop bad input with an error message of your own, to handle errors with `try` and `except`, and to write small tests that check whether a class does what its docstrings promise, starting with today's `Student`.

### 📌 This Week's Assignment
You will fit a neural network to the dropout data from Assignment 06 and compare it with the logistic regression in your `final_results`. Section 2.10 is the best preparation. I will post the assignment on Canvas and in the `Assignments/` folder of the course GitHub.

> 💡 **Tip:** When a class misbehaves, look at the object itself. `vars(obj)` shows the attributes stored on it right now, and printing it after each method call shows what that method changed. In a first class, the bug is usually a missing `self.` or an object built from an older version of the class.